In [1]:
import os
import gzip
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tabix

pd.options.mode.copy_on_write = True 

In [2]:
approach = 'attention'

# Extract layer information

```python
results:dict x8 keys (heads)
results[0]:list x200 tuples
results[0][0]:tuple x2 lists
results[0][0][0]:numpy.ndarray x1536 'scores'
results[0][0][1]:list x1536 'bins'
```

In [3]:
# Import sequences
sequences = pd.read_csv('/Enformer/enformer_sequences/sequences.csv')

# Create a dictionary with all the data
results_dict:dict = {}
for layer in range(11):
    with open(f'/Enformer/enformer_scores/examples_scores_{approach}_layer{layer}.p', 'rb') as f:
        results:dict = pickle.load(f)
    for head in range(8):
        for example in range(len(results[head])):
            gene:str = sequences['label'].iloc[example]
            if layer == 0 and head == 0:
                results_dict[gene] = {}
            results_dict[gene][f'layer{layer}-head{head}'] = results[head][example][0]
            
# Save the dictionary
with open(f'/Enformer/enformer_results/enformer_scores_dictionary_{approach}.pkl', 'wb') as f:
    pickle.dump(results_dict, f)

del(results, head, example, gene, layer, sequences, f, results_dict)

``` python
results_dict:dict x200 keys 'genes'
results_dict['EnsemblID_Gene']:dict x88 (layers[0-10]-heads[0-7])
results_dict['EnsemblID_Gene']['layer0-head0']:numpy.ndarray x1536 'bin scores'
```

# Analysis

``` python
results_dict:dict x200 keys 'genes'
results_dict['EnsemblID_Gene']:dict x88 (layers[0-10]-heads[0-7])
results_dict['EnsemblID_Gene']['layer0-head0']:numpy.ndarray x1536 'bin scores'
```

In [3]:
# Open sequence dictionary
with open(f'/Enformer/enformer_results/enformer_scores_dictionary_{approach}.pkl', 'rb') as f:
    results_dict:dict = pickle.load(f)

# Open Blat dataframe
sequences_df:pd.DataFrame = pd.read_csv(f'/Enformer/enformer_sequences/sequences.csv')

del(f)

In [ ]:
# Extract conservation information (PhyloP)
phyloP:tabix = tabix.open('/databases/conservation/hg38.phyloP100way.sorted.combined.bed.gz')
for _,row in sequences_df.iterrows():
    start, end = [int(i) for i in row['region'].split(':')[1].split('-')]
    ## Retrieve information from phyloP database with tabix
    records:object = phyloP.querys(f"chr{row['region']}")
    values:list = np.zeros(196608, dtype=np.float64)
    for record in records:
        for position in range(int(record[1]), int(record[2])):
            if position >= start and position < end:
                values[position-start] = float(record[3])
    values = values.reshape((1536, 128))
    values = np.mean(values, axis=1)
    ## Add this layer to the original one
    results_dict[row['label']]['phyloP'] = values

del(phyloP, start, end, records, record, values, position)

In [ ]:
# Annotate TSS regions
tss_db:tabix = tabix.open("/databases/refTSS4.1/refTSS_v4.1_human_coordinate.hg38.bed.txt.gz")
for _,row in sequences_df.iterrows():
    start, end = [int(i) for i in row['region'].split(':')[1].split('-')]
    ## Retrieve information from refTSS database with tabix
    records:object = tss_db.querys(f"chr{row['region']}")
    values:list = np.zeros(196608, dtype=np.float64)
    for record in records:
        for position in range(int(record[1]), int(record[2])):
            if position >= start and position < end:
                values[position-start] = 1
    values = values.reshape((1536, 128))
    values = np.mean(values, axis=1)
    ## Add this layer to the original one
    results_dict[row['label']]['TSS'] = values

del(tss_db, start, end, records, record, values, position)

In [ ]:
# Annotate enhancers
enhancers_db:tabix = tabix.open("/databases/SCREEN_Encode/K562.enhancers_hg38.bed.gz")
for _,row in sequences_df.iterrows():
    start, end = [int(i) for i in row['region'].split(':')[1].split('-')]
    ## Retrieve information from enhancers database with tabix
    records:object = enhancers_db.querys(f"chr{row['region']}")
    values:list = np.zeros(196608, dtype=np.float64)
    for record in records:
        for position in range(int(record[1]), int(record[2])):
            if position >= start and position < end:
                values[position-start] = 1
    values = values.reshape((1536, 128))
    values = np.mean(values, axis=1)
    ## Add this layer to the original one
    results_dict[row['label']]['enhancers'] = values

del(enhancers_db, start, end, records, record, values, position)

In [ ]:
# Annotate promoters
promoters_db:tabix = tabix.open("/databases/SCREEN_Encode/K562.promoters_hg38.bed.gz")
for _,row in sequences_df.iterrows():
    start, end = [int(i) for i in row['region'].split(':')[1].split('-')]
    ## Retrieve information from promoters database with tabix
    records:object = promoters_db.querys(f"chr{row['region']}")
    values:list = np.zeros(196608, dtype=np.float64)
    for record in records:
        for position in range(int(record[1]), int(record[2])):
            if position >= start and position < end:
                values[position-start] = 1
    values = values.reshape((1536, 128))
    values = np.mean(values, axis=1)
    ## Add this layer to the original one
    results_dict[row['label']]['promoters'] = values

del(promoters_db, start, end, records, record, values, position)

In [ ]:
# Annotate H3K4me3
H3K4me3_db:tabix = tabix.open("/databases/SCREEN_Encode/K562.H3K4me3_hg38.bed.gz")
for _,row in sequences_df.iterrows():
    start, end = [int(i) for i in row['region'].split(':')[1].split('-')]
    ## Retrieve information from H3K4me3 database with tabix
    records:object = H3K4me3_db.querys(f"chr{row['region']}")
    values:list = np.zeros(196608, dtype=np.float64)
    for record in records:
        for position in range(int(record[1]), int(record[2])):
            if position >= start and position < end:
                values[position-start] = 1
    values = values.reshape((1536, 128))
    values = np.mean(values, axis=1)
    ## Add this layer to the original one
    results_dict[row['label']]['H3K4me3'] = values

del(H3K4me3_db, start, end, records, record, values, position)

In [ ]:
# Annotate CTCF bound
ctcf_bound_db:tabix = tabix.open("/databases/SCREEN_Encode/K562.CTCF_only_hg38.bed.gz")
for _,row in sequences_df.iterrows():
    start, end = [int(i) for i in row['region'].split(':')[1].split('-')]
    ## Retrieve information from CTCF bound database with tabix
    records:object = ctcf_bound_db.querys(f"chr{row['region']}")
    values:list = np.zeros(196608, dtype=np.float64)
    for record in records:
        for position in range(int(record[1]), int(record[2])):
            if position >= start and position < end:
                values[position-start] = 1
    values = values.reshape((1536, 128))
    values = np.mean(values, axis=1)
    ## Add this layer to the original one
    results_dict[row['label']]['CTCF_bound'] = values

del(ctcf_bound_db, start, end, records, record, values, position)

In [ ]:
# Annotate CRISPR screening
crispr_screening_db:tabix = tabix.open("/databases/CRISPR_screen/enformer_ref9_CRISPR_K562_mmc2_enhancer_hg38.bed.gz")
for _,row in sequences_df.iterrows():
    start, end = [int(i) for i in row['region'].split(':')[1].split('-')]
    gene:str = row['label'].split('_')[0]
    ## Retrieve information from CRISPR screening database with tabix
    records:object = crispr_screening_db.querys(f"chr{row['region']}")
    values:list = np.zeros(196608, dtype=np.float64)
    values_hc:list = np.zeros(196608, dtype=np.float64)
    for record in records:
        if record[3] == gene and record[5] == 'TRUE':
            for position in range(int(record[1]), int(record[2])):
                if position >= start and position < end:
                    values[position-start] = 1
                    values_hc[position-start] = 1
        elif record[3] == gene:
            for position in range(int(record[1]), int(record[2])):
                if position >= start and position < end:
                    values[position-start] = 1
        else:
            continue
    values = values.reshape((1536, 128))
    values_hc = values_hc.reshape((1536, 128))
    values = np.mean(values, axis=1)
    values_hc = np.mean(values_hc, axis=1)
    ## Add this layer to the original one
    results_dict[row['label']]['CRISPR_screening'] = values
    results_dict[row['label']]['CRISPR_screening_high_confidence'] = values_hc

del(crispr_screening_db, start, end, records, record, values, values_hc, position)

In [ ]:
# Annotate CRISPR perturbation study 
crispr_perturbation_db:tabix = tabix.open("/databases/CRISPR_perturbations/ABC_predictions/K562/K562.Positive_Enhancer_Predictions_hg38.bed.gz")
for _,row in sequences_df.iterrows():
    start, end = [int(i) for i in row['region'].split(':')[1].split('-')]
    gene:str = row['label'].split('_')[1]
    ## Retrieve information from CRISPR perturbation study database with tabix
    records:object = crispr_perturbation_db.querys(f"chr{row['region']}")
    values:list = np.zeros(196608, dtype=np.float64)
    for record in records:
        if record[3] == gene:
            for position in range(int(record[1]), int(record[2])):
                if position >= start and position < end:
                    values[position-start] = 1
        else:
            continue
    values = values.reshape((1536, 128))
    values = np.mean(values, axis=1)
    ## Add this layer to the original one
    results_dict[row['label']]['CRISPR_perturbation'] = values

del(crispr_perturbation_db, start, end, records, record, values, position)

In [ ]:
# Annotate GC content
for _,row in sequences_df.iterrows():
    ## Change C and G to 1, and A and T to 0
    gc_sequence = row['sequence'].upper().replace('C', '1').replace('G', '1').replace('A', '0').replace('T', '0')
    values = np.array([int(i) for i in gc_sequence])
    
    ## Calculate the average in 128-mer windows
    values = values.reshape((1536, 128))
    values = np.mean(values, axis=1)
    ## Add this layer to the original one
    results_dict[row['label']]['GC'] = values
    
del(gc_sequence, values)

In [ ]:
# Extract TF information (JASPAR) -- It takes ~5m30s
tf_names = pd.read_csv('/databases/JASPAR/JASPAR_TF_families.csv')
jaspar:tabix = tabix.open('/databases/JASPAR/JASPAR_TFs_hg38.sorted.bed.gz')
for _,row in sequences_df.iterrows():
    start, end = [int(i) for i in row['region'].split(':')[1].split('-')]
    ## Create a list for each TF
    values_dict:dict = {}
    for _,tf in tf_names.iterrows():
        values_dict[tf['TF']] = np.zeros(196608, dtype=np.float64)
    ## Retrieve information from refTSS database with tabix
    records:object = jaspar.querys(f"chr{row['region']}")
    for record in records:
        for position in range(int(record[1]), int(record[2])):
            if position >= start and position < end:
                values_dict[str(record[3])][position-start] = 1
    ## Process each TF array
    for _,tf in tf_names.iterrows():
        values = values_dict[tf['TF']].reshape((1536, 128))
        values = np.mean(values, axis=1)
        ## Add this layer to the original one
        results_dict[row['label']][tf['TF']] = values

del(jaspar, row, start, end, values, values_dict, tf, records, record, position)

In [ ]:
# Combine TF matrix by families
uniqueTF_families:list = list(tf_names['Family'].unique())
for label in sequences_df['label']:
    for family in uniqueTF_families:
        idx:pd.Index = tf_names[tf_names['Family']==family].index
        results_dict[label][family] = np.nansum(np.vstack([results_dict[label][tf_names['TF'][i]] for i in idx]), axis=0)
    
del(uniqueTF_families, label, family, idx)

In [ ]:
# Remove TFs from the sequence dictionary
for label in sequences_df['label']:
    for tf in tf_names['TF']:
        try:
            results_dict[label].pop(tf)
        except KeyError:
            pass

del(tf_names, label, tf)

In [ ]:
# Extract repeated elements (RepeatMasker)
repeat_db:tabix = tabix.open("/databases/repeatmasker/repeat_masker_hg38.bed.gz")
repeat_families:pd.DataFrame = pd.read_csv('/databases/repeatmasker/repeat_families.csv', sep=",")
for _,row in sequences_df.iterrows():
    start, end = [int(i) for i in row['region'].split(':')[1].split('-')]
    ## Create a list for each repeat element
    values_dict:dict = {}
    for repeat in repeat_families['Family']:
        values_dict[repeat] = np.zeros(196608, dtype=np.float64)
    ## Retrieve information from repeat masker database with tabix
    records:object = repeat_db.querys(f"{row['region']}")
    for record in records:
        for position in range(int(record[1]), int(record[2])):
            if position >= start and position < end:
                try:
                    values_dict[str(record[4])][position-start] = 1
                except KeyError:
                    continue
    ## Process each repeat element array
    for repeat in repeat_families['Family']:
        values = values_dict[repeat].reshape((1536, 128))
        values = np.mean(values, axis=1)
        ## Add this layer to the original one
        results_dict[row['label']][repeat] = values

del(repeat_db, row, start, end, values, values_dict, repeat, records, record, position)

In [ ]:
# Combine repeats by families
unique_repeat_families:list = list(repeat_families['Superfamily'].unique())
for gene_id in results_dict:
    for family in unique_repeat_families:
        idx:pd.Index = repeat_families[repeat_families['Superfamily']==family].index
        results_dict[gene_id][family] = np.nansum(np.vstack([results_dict[gene_id][repeat_families['Family'][i]] for i in idx]), axis=0)
            
del(gene_id, family, idx, unique_repeat_families)

In [ ]:
# Remove single repeats from the results dictionary
for gene_id in results_dict:
    for repeat in repeat_families['Family']:
        try:
            results_dict[gene_id].pop(repeat)
        except KeyError:
            continue

del(gene_id, repeat, repeat_families)

In [ ]:
# Add positional features
for gene_id in results_dict:
    results_dict[gene_id]['position'] = np.array([i for i in range(0, 1536*128, 128)])
    results_dict[gene_id]['position_start'] = np.array([1 if i <= 48960 else 0 for i in range(0, 1536*128, 128)])
    results_dict[gene_id]['position_start_middle'] = np.array([1 if (i > 48960 and i <= 97920) else 0 for i in range(0, 1536*128, 128)])
    results_dict[gene_id]['position_middle_end'] = np.array([1 if (i > 97920 and i <= 146880) else 0 for i in range(0, 1536*128, 128)])
    results_dict[gene_id]['position_end'] = np.array([1 if i > 146880 else 0 for i in range(0, 1536*128, 128)])

del(gene_id)

In [ ]:
# Extract feature list to check
features_list:list = list(results_dict[list(results_dict.keys())[0]].keys())
features_list = [i for i in features_list if (i not in ['gene', 'phyloP', 'TSS', 'GC']) and (not i.startswith('position')) and (not i.startswith('layer'))]

# Initialize the feature dict
features_dict:dict = {}
for feature in results_dict[list(results_dict.keys())[0]]:
    if feature not in []:
        features_dict[feature] = []

# Filter features present in < 5% of sequences
remove_features:list = []
for key in results_dict:
    for feature in features_list:
        features_dict[feature].append(0 if max(results_dict[key][feature]) == 0 else 1)
for feature in features_list:
    if (sum(features_dict[feature]) / len(features_dict[feature])) < 0.05:
        remove_features.append(feature)

# Remove lowly expressed features
for key in results_dict:
    for feature2remove in remove_features:
        del results_dict[key][feature2remove]

del(features_list, features_dict, feature, remove_features, key, feature2remove)

In [ ]:
with open(f'/Enformer/enformer_results/enformer_scores_dictionary_{approach}_annotations.pkl', 'wb') as output:
    pickle.dump(results_dict, output, protocol=pickle.HIGHEST_PROTOCOL)

del(output)

# Prepare results for correlation analysis

In [ ]:
# Open previous results
## Enformer scores
with open(f'/Enformer/enformer_results/enformer_scores_dictionary_{approach}_annotations.pkl', 'rb') as f:
    results_dict:dict = pickle.load(f)

# Convert dictionary into list of dictionaries
df:list = []
for key, subdict in results_dict.items():
    dict_by_row:dict = {'gene': key}
    for subkey, values_list in subdict.items():
        dict_by_row[subkey] = ','.join(map(str, values_list))
    df.append(dict_by_row)

# Create DataFrame
df:pd.DataFrame = pd.DataFrame(df)

# Save the dataframe as a csv file with semicolon separator
# df.to_csv(f'/DNABERT/enhancer_results/enhancer_scores_dictionary_{approach}_updated_annotations_corformat.csv', sep=';', index=False)
with gzip.open(f'/Enformer/enformer_results/enformer_scores_dictionary_{approach}_annotations_corformat.csv.gz', 'wt', newline='', encoding='utf-8') as f:
    df.to_csv(f, sep=';', index=False)

del(key, subdict, subkey, dict_by_row, values_list, df, f)